<a href="https://colab.research.google.com/github/JuliBakagianni/delta_meth/blob/main/run_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sentence_transformers
import transformers
import torch
import sklearn
import yaml

import torch
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())

torch version: 2.10.0+cu128
cuda available: True
Device count: 1


---

4) Clone repository & checkout branch

Provide a `GITHUB_URL` variable to clone the repository into `/content/delta_meth`. Uncomment and set your GitHub URL as needed.

---

In [ ]:
from pathlib import Path
import zipfile, os
repo_path = '/content/delta_meth'
# clone repo if not present
if not Path(repo_path).exists():
    print('Cloning repository...')
    os.system(f'git clone https://github.com/JuliBakagianni/delta_meth.git {repo_path}')
# interactive upload if segments.zip not present
if not Path('segments.zip').exists():
    try:
        from google.colab import files
        print('Please upload segments.zip (select file in the dialog)')
        uploaded = files.upload()
        for fn, data in uploaded.items():
            open(fn, 'wb').write(data)
    except Exception as e:
        print('Upload skipped or not available:', e)
# ensure target dir exists and extract if zip present
target = Path(repo_path) / 'data' / 'processed' / 'segments'
target.mkdir(parents=True, exist_ok=True)
if Path('segments.zip').exists():
    print('Extracting segments.zip to', target)
    with zipfile.ZipFile('segments.zip', 'r') as z:
        z.extractall(target)
    print('Extraction complete.')
    extracted_files = list(target.glob('**/*')) # List all files recursively
    print(f'Found {len(extracted_files)} files after extraction:')
else:
    print('No segments.zip found; ensure JSONs are placed in', target)

---

7) Write pipeline configuration (YAML/JSON)

Load and print the active configuration from `configs/config.yaml` in the cloned repo. If the file is not present, show a default config and write it to disk.

---

In [ ]:
# Load config
import yaml
from pathlib import Path

cfg_path = Path('/content/delta_meth/configs/config.yaml')
cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8'))
print('Loaded config:')
print(cfg)

Loaded config:
{'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'nli_model': 'joeddav/xlm-roberta-large-xnli', 'chunk_size': 150, 'similarity_threshold': 0.5, 'nli_threshold': 0.7, 'llm_model': 'openai/gpt-4o-mini'}


### Segmentation evaluation

In [ ]:
import zipfile
import json
import re
from pathlib import Path

# Paths
raw_zip = '/content/evaggelismos_raw_txt_notes.zip'
raw_dir = Path('/content/raw_notes_eval')
seg_dir = Path('/content/delta_meth/data/processed/segments')

# Extract raw notes if needed
if Path(raw_zip).exists() and not raw_dir.exists():
    print(f'Extracting {raw_zip} to {raw_dir}...')
    with zipfile.ZipFile(raw_zip, 'r') as z:
        z.extractall(raw_dir)

# Collect raw and processed files
raw_files = {p.stem: p for p in raw_dir.glob('**/*.txt')} if raw_dir.exists() else {}
processed_files = {p.stem: p for p in seg_dir.glob('**/*.json')}

print(f'Found {len(raw_files)} raw notes and {len(processed_files)} processed JSON files.')

# Evaluation metrics
stats = {
    'total_evaluated': 0,
    'single_segment': 0,
    'multi_segment': 0,
    'multi_segment_correct_start': 0,
    'missing_raw_file': 0,
    'text_length_mismatch': 0,
    'empty_segments': 0
}

# Track anomalies
anomalies = []
empty_segment_notes = []

# Simple regex to guess if dates exist in raw text (adjust based on your actual date format)
date_pattern = re.compile(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}-\d{2}-\d{2}\b')

for stem, json_path in processed_files.items():
    if stem not in raw_files:
        stats['missing_raw_file'] += 1
        continue

    stats['total_evaluated'] += 1
    raw_text = raw_files[stem].read_text(encoding='utf-8')

    try:
        data = json.loads(json_path.read_text(encoding='utf-8'))
        segs = data.get('segments', [])
    except Exception as e:
        print(f'Error reading {json_path}: {e}')
        continue

    seg_len = len(segs)
    total_seg_text_len = 0

    for idx, s in enumerate(segs):
        text = s.get('text', '')
        total_seg_text_len += len(text)
        if not text.strip():
            stats['empty_segments'] += 1
            empty_segment_notes.append({'stem': stem, 'date': s.get('date', 'Unknown'), 'index': idx})

    raw_text_len = len(raw_text)

    # Check length mismatch (allowing 10% tolerance for headers/whitespace removal)
    if raw_text_len > 0 and (total_seg_text_len / raw_text_len < 0.90 or total_seg_text_len / raw_text_len > 1.10):
        stats['text_length_mismatch'] += 1

    if seg_len == 1:
        stats['single_segment'] += 1
        # Optional: check if raw_text actually contains dates. If it does, maybe it should have been segmented.
    elif seg_len > 1:
        stats['multi_segment'] += 1
        # Check if first segment is 'pre ICU' / 'before_ICU'
        first_date = str(segs[0].get('date', '')).lower()
        if 'before' in first_date or 'pre' in first_date:
            stats['multi_segment_correct_start'] += 1
        else:
            anomalies.append({'stem': stem, 'first_date': segs[0].get('date', 'Unknown')})

print('\n--- Segmentation Evaluation Results ---')
for k, v in stats.items():
    print(f'{k}: {v}')

if stats['multi_segment'] > 0:
    correct_start_pct = (stats['multi_segment_correct_start'] / stats['multi_segment']) * 100
    print(f'\nMulti-segment notes correctly starting with pre-ICU: {correct_start_pct:.1f}%')

    if empty_segment_notes:
        print(f'\n--- Found {len(empty_segment_notes)} empty segments ---')
        for a in empty_segment_notes[:15]:
            print(f"Note ID: {a['stem']} -> Segment Index: {a['index']}, Date: '{a['date']}'")
        if len(empty_segment_notes) > 15:
            print(f"... and {len(empty_segment_notes) - 15} more.")
    else:
        print('\n--- Good news: No empty segments found! ---')

    if anomalies:
        print(f'\n--- Investigating {len(anomalies)} multi-segment notes without pre-ICU start ---')
        for a in anomalies[:15]: # Show up to 15 examples
            print(f"Note ID: {a['stem']} -> Starts with Date: '{a['first_date']}'")
        if len(anomalies) > 15:
            print(f"... and {len(anomalies) - 15} more.")


Found 574 raw notes and 287 processed JSON files.

--- Segmentation Evaluation Results ---
total_evaluated: 287
single_segment: 131
multi_segment: 156
multi_segment_correct_start: 137
missing_raw_file: 0
text_length_mismatch: 1
empty_segments: 5

Multi-segment notes correctly starting with pre-ICU: 87.8%

--- Found 5 empty segments ---
Note ID: 25363214 -> Segment Index: 1, Date: '2024-06-03'
Note ID: 21363910 -> Segment Index: 29, Date: '2024-05-13'
Note ID: 25426210 -> Segment Index: 4, Date: '2024-09-09'
Note ID: 25477377 -> Segment Index: 12, Date: '26/11'
Note ID: 25477377 -> Segment Index: 13, Date: '27/11'

--- Investigating 19 multi-segment notes without pre-ICU start ---
Note ID: 23458098 -> Starts with Date: '22/11'
Note ID: 23458098Γò¼ΓûÆ -> Starts with Date: '25/10-21'
Note ID: 25332587 -> Starts with Date: '2024-04-21'
Note ID: 24954092 -> Starts with Date: '02-17/07'
Note ID: 20097899 -> Starts with Date: '11/03'
Note ID: 25349980 -> Starts with Date: '2024-05-22'
Note ID

---

### Call pipeline functions interactively

This cell imports `run_pipeline` and runs sequential comparisons for a single note's segments so you can inspect intermediate outputs. If model loading fails, a simple mock fallback will be used for demonstration.

In [6]:
import sys, json, glob
from pathlib import Path
repo_path = '/content/delta_meth'
if repo_path not in sys.path: sys.path.insert(0, repo_path)
from src.pipeline.run_pipeline import run_pipeline


# list available segment JSONs
seg_dir = Path(repo_path) / 'data' / 'processed' / 'segments'
files = sorted([p for p in seg_dir.glob('**/*.json')]) if seg_dir.exists() else [] # Modified to look for JSONs recursively
print('Found', len(files), 'segment files')
# pick first multi-segment file for interactive inspection
sample = None
for p in files:
    data = json.loads(p.read_text(encoding='utf-8'))
    if len(data.get('segments', [])) > 1:
        sample = p
        break
if not sample:
    print('No multi-segment file found; add JSONs to', seg_dir)
else:
    print('Using sample file:', sample)
    note = json.loads(sample.read_text(encoding='utf-8'))
    segs = note.get('segments', [])
    print('Segment count:', len(segs))
    # sequentially compare segment[i] vs segment[i+1]
    for i in range(len(segs)-1):
        a = segs[i].get('text','')
        b = segs[i+1].get('text','')
        print('Comparison', i, 'dates:', segs[i].get('date'), 'vs', segs[i+1].get('date'))
        try:
            contradiction, detailed = run_pipeline(note_a=a, note_b=b, config_path=str(Path(repo_path)/'configs'/'config.yaml'), verbose=True, chunk_unit='sentences', chunk_size=1)
            print('Contradiction:', contradiction)
        except Exception as e:
            print('Real pipeline failed:', e)
            # Mock fallback: simple heuristic demo
            from src.preprocessing.chunking import chunk_notes
            ca = chunk_notes(a, chunk_size=1, chunk_unit='sentences')
            cb = chunk_notes(b, chunk_size=1, chunk_unit='sentences')
            detailed = []
            for idx, sent in enumerate(ca):
                other = cb[idx] if idx < len(cb) else (cb[-1] if cb else '')
                label = 'contradiction' if ('πυρετό' in sent and 'πυρετό' not in other) or ('πυρετό' in other and 'πυρετό' not in sent) else 'neutral'
                detailed.append({'i': idx, 'j': idx if idx < len(cb) else len(cb)-1, 'nli_label': label, 'chunk1': sent, 'chunk2': other})
            contradictions = [d for d in detailed if d['nli_label']=='contradiction']
            contradiction = contradictions[0] if contradictions else None
            print('Mock contradiction:', contradiction)

Found 287 segment files
Using sample file: /content/delta_meth/data/processed/segments/segments/20052347.json
Segment count: 3
Comparison 0 dates: before_ICU vs 2024-01-23
[pipeline] note A -> 6 chunks
[pipeline] note B -> 5 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[pipeline] aligned candidate pairs (sim>=0.5): 4
[pipeline] pair (0,6) sim=0.560 nli=entailment (0.995)
  A: Η ασθενής μεταφέρεται στη ΜΕΘ μετά τη θρομβεκτομή για παρακολούθηση
  B: Mεταφέρεται στη νευρολογική κλινική για συνέχιση νοσηλείας
[pipeline] pair (1,0) sim=0.578 nli=entailment (0.914)
  A: Σε αυτόματη αναπνοή σε VM50% με καλή αεριομετρική εικόνα
  B: Ιδιες συνθήκες αναπνευστικά
[pipeline] pair (2,3) sim=0.540 nli=entailment (0.512)
  A: Αιμοδυναμικά σταθερή
  B: Διούρηση ικανοποιητική
[pipeline] pair (4,1) sim=0.736 nli=contradiction (0.993)
  A: GCS: 12/15 (δεν ανοίγει τους οφθαλμούς), ΑΡ πυραμιδική συνδρομή- εκτελεί με ΔΕ άνω-κάτω άκρο, ομιλεί
  B: GCS: 14/15 (ανοίγει τους οφθαλμούς στα παραγγέλματα, εκτελεί με ΔΕ άνω-κάτω άκρο, ομιλεί)
[pipeline] contradiction FOUND:
  sim_score=0.736
  nli_confidence=0.993
  chunk A:
    GCS: 12/15 (δεν ανοίγει τους οφθαλμούς), ΑΡ πυραμιδική συνδρομή- εκτελεί με ΔΕ άνω-κάτω άκρο, ομιλεί
  chunk B:
    GCS: 14/15 (ανοίγει τους οφθαλμούς στ

---

### Batch run (optional)

If you want to run the diagnostic comparisons across many files, uncomment and run the cell below. Beware: this will load models for each comparison and may be slow or require more RAM/GPU.

In [ ]:
# Example batch loop (commented out by default)
# from pathlib import Path
# repo_path = '/content/delta_meth'
# seg_dir = Path(repo_path)/'data'/'processed'/'segments'
# out_dir = Path(repo_path)/'data'/'results'/'diagnostic_shifts'
# out_dir.mkdir(parents=True, exist_ok=True)
# files = sorted([p for p in seg_dir.glob('*.json')])
# for p in files[:50]:  # limit to first 50 for demo
#     note = json.loads(p.read_text(encoding='utf-8'))
#     segs = note.get('segments', [])
#     if len(segs)<=1:
#         continue
#     results = []
#     for i in range(len(segs)-1):
#         a = segs[i].get('text','')
#         b = segs[i+1].get('text','')
#         contradiction, detailed = run_pipeline(note_a=a, note_b=b, config_path=str(Path(repo_path)/'configs'/'config.yaml'), verbose=False, chunk_unit='sentences', chunk_size=1)
#         results.append({'i': i, 'contradiction': contradiction})
#     out_path = out_dir / (p.stem + '.diagnostic_shifts.json')
#     out_path.write_text(json.dumps({'note_id': p.stem, 'comparisons': results}, ensure_ascii=False, indent=2), encoding='utf-8')
# print('Batch run complete')